In [ ]:
# Cell 1: Change to the correct directory
import os
from repo_paths import bootstrap, REPO_ROOT, FROZEN_INST
bootstrap()
print("Current directory:", os.getcwd())

# Cell 2: Add the directory to Python path (if needed)
import sys
sys.path.insert(0, '.')  # Add current directory
sys.path.append('jax_ib/')  # Add jax_ib path
sys.path.insert(0, 'jax-cfd')  # Add jax-cfd path
print("Python path updated")


# Cell 3: Now import should work
from channel_obstacle_flow import *
print("✅ Successfully imported obstacle flow demo!")

from channel_constriction_flow import *
print("✅ Successfully imported constriction flow demo!")

from porous_media_flow import *
print("✅ Successfully imported porous media flow demo!")

#from serpentine_flow import *
#print("✅ Successfully imported serpentine flow demo!")

import jax
jax.config.update('jax_enable_x64', False)

In [ ]:
import jax

# Test if JAX is connected to a GPU
if jax.default_backend() == 'gpu':
    print("✅ JAX is connected to a GPU!")
else:
    print("⚠️ JAX is NOT connected to a GPU. Current backend:", jax.default_backend())


In [ ]:
# That's it! Now you can run:
run_channel_obstacle('newtonian', 1.0)
# run_channel_obstacle('power_law', 0.5, 0.8) 
# run_channel_obstacle('carreau_yasuda', 0.02, 0.5, 5.0, 0.8, 2.0)

In [ ]:
run_channel_obstacle('power_law', 0.5, 0.8) 
# run_channel_obstacle('carreau_yasuda', 0.02, 0.5, 5.0, 0.8, 2.0)

In [ ]:
run_channel_obstacle('carreau_yasuda', 0.02, 1.0, 5.0, 0.7, 2.0)

In [ ]:
run_channel_obstacle('carreau_yasuda', 0.02, 1.0, 5.0, 0.6, 2.0, preconditioner='helmholtz', dt = 5e-5, outer_steps = 1200)

In [ ]:
run_channel_obstacle('carreau_yasuda', 0.02, 1.0, 5.0, 0.6, 2.0, preconditioner='helmholtz', dt = 5e-5, outer_steps = 1200)

In [ ]:
run_channel_obstacle('carreau_yasuda', 0.02, 1.0, 5.0, 0.6, 2.0, pressure_gradient = 25.0, preconditioner='helmholtz', dt = 5e-6, outer_steps = 1800)

In [ ]:
run_channel_obstacle('carreau_yasuda', 0.05, 2.0, 10.0, 0.6, 2.0, pressure_gradient = 100.0, preconditioner='helmholtz', dt = 2e-6, outer_steps = 2000)

In [ ]:
# Import necessary functions
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from jax_rheology.models import carreau_yasuda_viscosity

# Fixed Carreau-Yasuda parameters
eta_inf = 0.02  # Infinite-shear viscosity
eta_0 = 1.0     # Zero-shear viscosity
lambda_ = 5.0   # Time constant
a = 2.0         # Shape parameter

# Variable parameter: n (power-law index)
n_values = np.arange(0.6, 1.0, 0.1)

# Create strain rate range (logarithmic scale)
strain_rates = jnp.logspace(-3, 2, 1000)  # 0.001 to 100 s^-1

# Plot setup
plt.figure(figsize=(10, 7))
colors = plt.cm.viridis(np.linspace(0, 1, len(n_values)))

# Plot viscosity curves for each n value
for i, n in enumerate(n_values):
    viscosity = carreau_yasuda_viscosity(strain_rates, eta_inf, eta_0, lambda_, n, a)
    plt.loglog(strain_rates, viscosity, color=colors[i], linewidth=2.5, 
               label=f'n = {n:.1f}')

# Add horizontal lines for reference
plt.axhline(eta_0, color='gray', linestyle='--', alpha=0.5, label=f'η₀ = {eta_0}')
plt.axhline(eta_inf, color='gray', linestyle=':', alpha=0.5, label=f'η∞ = {eta_inf}')

# Formatting
plt.xlabel('Strain Rate (s⁻¹)', fontsize=12)
plt.ylabel('Viscosity (Pa·s)', fontsize=12)
plt.title(f'Carreau-Yasuda Viscosity Curves\n(η∞={eta_inf}, η₀={eta_0}, λ={lambda_}, a={a})', fontsize=14)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.xlim(1e-3, 1e2)
plt.ylim(eta_inf/2, eta_0*2)

# Add annotation
plt.text(0.02, 0.95, 'Shear-thinning behavior\n(decreasing n → more thinning)', 
         transform=plt.gca().transAxes, fontsize=10, 
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
         verticalalignment='top')

plt.tight_layout()
plt.show()

# Print summary
print("Carreau-Yasuda Parameters:")
print(f"  η∞ (infinite-shear viscosity): {eta_inf}")
print(f"  η₀ (zero-shear viscosity): {eta_0}") 
print(f"  λ (time constant): {lambda_}")
print(f"  a (shape parameter): {a}")
print(f"  n (power-law index): {n_values} (varied)")
print(f"\nFormula: η(γ̇) = η∞ + (η₀-η∞)[1+(λγ̇)ᵃ]^((n-1)/a)")

In [ ]:
# Import necessary functions
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from jax_rheology.models import carreau_yasuda_viscosity

# Fixed Carreau-Yasuda parameters
eta_inf = 0.02  # Infinite-shear viscosity
eta_0 = 1.0     # Zero-shear viscosity
n = 0.7         # Power-law index (fixed)
a = 2.0         # Shape parameter (fixed)

# Variable parameter: lambda (time constant)
lambda_values = np.array([0.5, 1.0, 2.0, 5.0, 10.0, 20.0])

# Create strain rate range (logarithmic scale)
strain_rates = jnp.logspace(-3, 2, 1000)  # 0.001 to 100 s^-1

# Plot setup
plt.figure(figsize=(10, 7))
colors = plt.cm.plasma(np.linspace(0, 1, len(lambda_values)))

# Plot viscosity curves for each lambda value
for i, lambda_ in enumerate(lambda_values):
    viscosity = carreau_yasuda_viscosity(strain_rates, eta_inf, eta_0, lambda_, n, a)
    plt.loglog(strain_rates, viscosity, color=colors[i], linewidth=2.5, 
               label=f'λ = {lambda_}')

# Add horizontal lines for reference
plt.axhline(eta_0, color='gray', linestyle='--', alpha=0.5, label=f'η₀ = {eta_0}')
plt.axhline(eta_inf, color='gray', linestyle=':', alpha=0.5, label=f'η∞ = {eta_inf}')

# Formatting
plt.xlabel('Strain Rate (s⁻¹)', fontsize=12)
plt.ylabel('Viscosity (Pa·s)', fontsize=12)
plt.title(f'Carreau-Yasuda: Varying Time Constant λ\n(η∞={eta_inf}, η₀={eta_0}, n={n}, a={a})', fontsize=14)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.xlim(1e-3, 1e2)
plt.ylim(eta_inf/2, eta_0*2)

# Add annotation
plt.text(0.02, 0.95, 'Higher λ → transition at lower strain rates\nLower λ → transition at higher strain rates', 
         transform=plt.gca().transAxes, fontsize=10, 
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
         verticalalignment='top')

plt.tight_layout()
plt.show()

# Print summary
print("Carreau-Yasuda Parameters (varying λ):")
print(f"  η∞: {eta_inf} (fixed)")
print(f"  η₀: {eta_0} (fixed)") 
print(f"  λ: {lambda_values} (varied)")
print(f"  n: {n} (fixed)")
print(f"  a: {a} (fixed)")

In [ ]:
# Import necessary functions
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from jax_rheology.models import carreau_yasuda_viscosity

# Fixed Carreau-Yasuda parameters
eta_inf = 0.02  # Infinite-shear viscosity
eta_0 = 1.0     # Zero-shear viscosity
n = 0.7         # Power-law index (fixed)
lambda_ = 5.0   # Time constant (fixed)

# Variable parameter: a (shape parameter)
a_values = np.array([0.5, 1.0, 1.5, 2.0, 3.0, 5.0])

# Create strain rate range (logarithmic scale)
strain_rates = jnp.logspace(-3, 2, 1000)  # 0.001 to 100 s^-1

# Plot setup
plt.figure(figsize=(10, 7))
colors = plt.cm.coolwarm(np.linspace(0, 1, len(a_values)))

# Plot viscosity curves for each a value
for i, a in enumerate(a_values):
    viscosity = carreau_yasuda_viscosity(strain_rates, eta_inf, eta_0, lambda_, n, a)
    plt.loglog(strain_rates, viscosity, color=colors[i], linewidth=2.5, 
               label=f'a = {a}')

# Add horizontal lines for reference
plt.axhline(eta_0, color='gray', linestyle='--', alpha=0.5, label=f'η₀ = {eta_0}')
plt.axhline(eta_inf, color='gray', linestyle=':', alpha=0.5, label=f'η∞ = {eta_inf}')

# Formatting
plt.xlabel('Strain Rate (s⁻¹)', fontsize=12)
plt.ylabel('Viscosity (Pa·s)', fontsize=12)
plt.title(f'Carreau-Yasuda: Varying Shape Parameter a\n(η∞={eta_inf}, η₀={eta_0}, λ={lambda_}, n={n})', fontsize=14)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.xlim(1e-3, 1e2)
plt.ylim(eta_inf/2, eta_0*2)

# Add annotation
plt.text(0.02, 0.95, 'Lower a → sharper transition\nHigher a → more gradual transition', 
         transform=plt.gca().transAxes, fontsize=10, 
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
         verticalalignment='top')

plt.tight_layout()
plt.show()

# Print summary
print("Carreau-Yasuda Parameters (varying a):")
print(f"  η∞: {eta_inf} (fixed)")
print(f"  η₀: {eta_0} (fixed)") 
print(f"  λ: {lambda_} (fixed)")
print(f"  n: {n} (fixed)")
print(f"  a: {a_values} (varied)")

In [ ]:
# Import necessary functions
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from jax_rheology.models import carreau_yasuda_viscosity

# Fixed Carreau-Yasuda parameters
eta_inf = 0.01  # Infinite-shear viscosity
eta_0 = 1.0     # Zero-shear viscosity
n = 0.6         # Power-law index (fixed)
lambda_ = 5.0   # Time constant (fixed)

# Variable parameter: a (shape parameter)
a_values = np.array([0.5, 1.0, 1.5, 2.0, 3.0, 5.0])

# Create strain rate range (logarithmic scale)
strain_rates = jnp.logspace(-3, 2, 1000)  # 0.001 to 100 s^-1

# Plot setup
plt.figure(figsize=(10, 7))
colors = plt.cm.coolwarm(np.linspace(0, 1, len(a_values)))

# Plot viscosity curves for each a value
for i, a in enumerate(a_values):
    viscosity = carreau_yasuda_viscosity(strain_rates, eta_inf, eta_0, lambda_, n, a)
    plt.loglog(strain_rates, viscosity, color=colors[i], linewidth=2.5, 
               label=f'a = {a}')

# Add horizontal lines for reference
plt.axhline(eta_0, color='gray', linestyle='--', alpha=0.5, label=f'η₀ = {eta_0}')
plt.axhline(eta_inf, color='gray', linestyle=':', alpha=0.5, label=f'η∞ = {eta_inf}')

# Formatting
plt.xlabel('Strain Rate (s⁻¹)', fontsize=12)
plt.ylabel('Viscosity (Pa·s)', fontsize=12)
plt.title(f'Carreau-Yasuda: Varying Shape Parameter a\n(η∞={eta_inf}, η₀={eta_0}, λ={lambda_}, n={n})', fontsize=14)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.xlim(1e-3, 1e2)
plt.ylim(eta_inf/2, eta_0*2)

# Add annotation
plt.text(0.02, 0.95, 'Lower a → sharper transition\nHigher a → more gradual transition', 
         transform=plt.gca().transAxes, fontsize=10, 
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
         verticalalignment='top')

plt.tight_layout()
plt.show()

# Print summary
print("Carreau-Yasuda Parameters (varying a):")
print(f"  η∞: {eta_inf} (fixed)")
print(f"  η₀: {eta_0} (fixed)") 
print(f"  λ: {lambda_} (fixed)")
print(f"  n: {n} (fixed)")
print(f"  a: {a_values} (varied)")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Parameters You Can Change ---

# Amplitude: Controls the height of the wave's centerline
amplitude = 1.0

# Angular Frequency: Controls the period/length of the wave. Smaller is longer.
omega = 1.0

# Channel Width: The total width of the channel is 2 * half_width
half_width = 0.5

# --- Data Generation ---

# Create 500 x-points from -2*pi to 2*pi
x_points = np.linspace(-2 * np.pi, 2 * np.pi, 500)

# Calculate the y-points for the centerline using a simple sine wave
centerline_y = amplitude * np.sin(omega * x_points)

# Calculate the y-points for the top and bottom walls
top_wall_y = centerline_y + half_width
bottom_wall_y = centerline_y - half_width

# --- Plotting the Data (Optional) ---

# This part visualizes the arrays
plt.plot(x_points, top_wall_y, color='blue')
plt.plot(x_points, bottom_wall_y, color='blue')
plt.axis('equal')
plt.grid(True)
plt.title('Sine Wave Channel')
plt.show()

In [ ]:
run_channel_constriction('newtonian', 1.0)

In [ ]:
run_channel_constriction('carreau_yasuda', 0.02, 1.0, 5.0, 0.6, 2.0, preconditioner='helmholtz', dt = 5e-5, outer_steps = 1200)

In [ ]:
result = run_channel_constriction('carreau_yasuda', 0.02, 1.0, 5.0, 0.6, 2.0, pressure_gradient = 20.0, preconditioner='helmholtz', dt = 2e-6, outer_steps = 2500)

In [ ]:
result = run_channel_constriction('carreau_yasuda', 0.02, 1.0, 20.0, 0.6, 2.0, pressure_gradient = 20.0, preconditioner='helmholtz', dt = 1e-6, outer_steps = 2500)

In [ ]:
result = run_channel_constriction('carreau_yasuda', 0.02, 1.0, 5.0, 0.5, 2.0, pressure_gradient = 20.0, preconditioner='helmholtz', dt = 1e-6, outer_steps = 5000)

In [ ]:
result = run_channel_constriction('carreau_yasuda', 0.02, 1.0, 5.0, 0.5, 2.0, pressure_gradient = 5.0, preconditioner='none', dt = 2e-5, outer_steps = 200)

In [ ]:
result = run_channel_constriction('carreau_yasuda', 0.02, 1.0, 5.0, 0.5, 2.0, pressure_gradient = 5.0, preconditioner='none', dt = 1e-5, inner_steps = 200, outer_steps = 200)

In [ ]:
result = run_channel_constriction('carreau_yasuda', 0.02, 1.0, 5.0, 0.5, 2.0, pressure_gradient = 1.0, preconditioner='none', dt = 5e-5, outer_steps = 100)

In [ ]:
# Import necessary functions
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from jax_rheology.models import carreau_yasuda_viscosity

# Fixed Carreau-Yasuda parameters
eta_inf = 0.02  # Infinite-shear viscosity
eta_0 = 1.0     # Zero-shear viscosity
lambda_ = 5.0   # Time constant
a = 2.0         # Shape parameter

# Variable parameter: n (power-law index)
n_values = np.arange(0.5, 1.0, 0.1)

# Create strain rate range (logarithmic scale)
strain_rates = jnp.logspace(-3, 4, 1000)  # 0.001 to 100 s^-1

# Plot setup
plt.figure(figsize=(10, 7))
colors = plt.cm.viridis(np.linspace(0, 1, len(n_values)))

# Plot viscosity curves for each n value
for i, n in enumerate(n_values):
    viscosity = carreau_yasuda_viscosity(strain_rates, eta_inf, eta_0, lambda_, n, a)
    plt.loglog(strain_rates, viscosity, color=colors[i], linewidth=2.5, 
               label=f'n = {n:.1f}')

# Add horizontal lines for reference
plt.axhline(eta_0, color='gray', linestyle='--', alpha=0.5, label=f'η₀ = {eta_0}')
plt.axhline(eta_inf, color='gray', linestyle=':', alpha=0.5, label=f'η∞ = {eta_inf}')

# Formatting
plt.xlabel('Strain Rate (s⁻¹)', fontsize=12)
plt.ylabel('Viscosity (Pa·s)', fontsize=12)
plt.title(f'Carreau-Yasuda Viscosity Curves\n(η∞={eta_inf}, η₀={eta_0}, λ={lambda_}, a={a})', fontsize=14)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.xlim(1e-3, 1e2)
plt.ylim(eta_inf/2, eta_0*2)

# Add annotation
plt.text(0.02, 0.95, 'Shear-thinning behavior\n(decreasing n → more thinning)', 
         transform=plt.gca().transAxes, fontsize=10, 
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
         verticalalignment='top')

plt.tight_layout()
plt.show()

# Print summary
print("Carreau-Yasuda Parameters:")
print(f"  η∞ (infinite-shear viscosity): {eta_inf}")
print(f"  η₀ (zero-shear viscosity): {eta_0}") 
print(f"  λ (time constant): {lambda_}")
print(f"  a (shape parameter): {a}")
print(f"  n (power-law index): {n_values} (varied)")
print(f"\nFormula: η(γ̇) = η∞ + (η₀-η∞)[1+(λγ̇)ᵃ]^((n-1)/a)")

In [ ]:
# Import necessary functions
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from jax_rheology.models import carreau_yasuda_viscosity

# Define a simple Power-Law model function for comparison
def power_law_viscosity(strain_rate, K, n):
    """Calculates viscosity using the Power-Law model: η = K * γ̇^(n-1)"""
    return K * strain_rate**(n - 1)

# --- Parameters ---
# Fixed Carreau-Yasuda parameters
eta_0 = 1.0     # Zero-shear viscosity (Pa·s)
lambda_ = 5.0   # Time constant (s)
a = 2.0         # Shape parameter
n = 0.5         # Fixed power-law index

# Variable parameter: eta_inf (infinite-shear viscosity)
eta_inf_values = np.array([0.01, 0.05, 0.1, 0.2])

# Create strain rate range (logarithmic scale)
strain_rates = jnp.logspace(-3, 4, 1000)  # 0.001 to 10000 s⁻¹

# --- Plotting ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(12, 8))
colors = plt.cm.plasma(np.linspace(0, 0.8, len(eta_inf_values)))

# 1. Plot Carreau-Yasuda curves for each eta_inf value
for i, eta_inf in enumerate(eta_inf_values):
    viscosity = carreau_yasuda_viscosity(strain_rates, eta_inf, eta_0, lambda_, n, a)
    plt.loglog(strain_rates, viscosity, color=colors[i], linewidth=2.5, 
               label=f'Carreau-Yasuda (η∞ = {eta_inf})')

# 2. Plot the comparative Power-Law model
# We set the consistency index K to match the Carreau-Yasuda model in the power-law region
K = eta_0 * (lambda_ ** (n - 1))
power_law_visc = power_law_viscosity(strain_rates, K, n)
plt.loglog(strain_rates, power_law_visc, color='black', linestyle='--', linewidth=2, 
           label=f'Power-Law (K={K:.2f}, n={n:.1f})')

# --- Formatting & Annotations ---
# Add reference lines
plt.axhline(eta_0, color='gray', linestyle=':', alpha=0.8, label=f'η₀ = {eta_0}')
max_exp_strain_rate = 10 # Example max strain rate
plt.axvline(max_exp_strain_rate, color='red', linestyle='-.', alpha=0.7, 
            label=f'Max experimental γ̇ = {max_exp_strain_rate} s⁻¹')


# Add annotation explaining the key insight
plt.text(0.02, 0.98, 
         'Note how η∞ changes the curvature\nin the transition region, even before\nthe second Newtonian plateau is reached.', 
         transform=plt.gca().transAxes, fontsize=11, 
         bbox=dict(boxstyle='round,pad=0.5', facecolor='aliceblue', alpha=0.9),
         verticalalignment='top', fontweight='bold')

# Labels, Title, and Legend
plt.xlabel('Shear Rate, γ̇ (s⁻¹)', fontsize=13)
plt.ylabel('Viscosity, η (Pa·s)', fontsize=13)
plt.title(f'Effect of η∞ on Carreau-Yasuda Model vs. Power-Law\n(η₀={eta_0}, λ={lambda_}, n={n}, a={a})', fontsize=15, pad=10)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=11)
plt.grid(True, which="both", ls="-", alpha=0.2)
plt.xlim(1e-3, 1e4)
plt.ylim(eta_inf_values.min() / 2, eta_0 * 2)

plt.tight_layout(rect=[0, 0, 0.85, 1]) # Adjust layout to make space for legend
plt.show()

# --- Print Summary ---
print("Model Parameters:")
print("-" * 25)
print(f"  η₀ (zero-shear viscosity): {eta_0} (fixed)")
print(f"  λ (time constant):         {lambda_} (fixed)")
print(f"  a (shape parameter):         {a} (fixed)")
print(f"  n (power-law index):       {n} (fixed)")
print(f"  η∞ (inf-shear viscosity):  {eta_inf_values} (varied)")
print("\nCarreau-Yasuda Formula: η(γ̇) = η∞ + (η₀-η∞)[1+(λγ̇)ᵃ]^((n-1)/a)")
print(f"Power-Law Formula:    η(γ̇) = K * γ̇^(n-1), with K = {K:.3f}")

In [ ]:
# Import necessary functions
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from jax_rheology.models import carreau_yasuda_viscosity

# --- Parameters (same as before) ---
# Fixed Carreau-Yasuda parameters
eta_0 = 1.0     # Zero-shear viscosity (Pa·s)
lambda_ = 5.0   # Time constant (s)
a = 2.0         # Shape parameter
n = 0.5         # Fixed power-law index

# Variable parameter: eta_inf (infinite-shear viscosity)
# We focus on the values that will be visible in the zoomed plot
eta_inf_values = np.array([0.01, 0.025, 0.05, 0.075])

# Create strain rate range (linear scale, focused on the transition)
strain_rates = jnp.linspace(20, 300, 1000)  # Shear rates from 20 to 300 s⁻¹

# --- Plotting ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(12, 8))
colors = plt.cm.plasma(np.linspace(0, 0.8, len(eta_inf_values)))

# 1. Plot Carreau-Yasuda curves for each eta_inf value
for i, eta_inf in enumerate(eta_inf_values):
    viscosity = carreau_yasuda_viscosity(strain_rates, eta_inf, eta_0, lambda_, n, a)
    plt.plot(strain_rates, viscosity, color=colors[i], linewidth=2.5,
             label=f'Carreau-Yasuda (η∞ = {eta_inf})')
    # Add a horizontal line for the plateau of each curve
    plt.axhline(eta_inf, color=colors[i], linestyle='--', alpha=0.6)


# --- Formatting & Annotations ---
# Add annotation explaining the key insight
#plt.text(0.98, 0.98,
 #        'On a linear scale, the curves clearly\nseparate and approach their unique\ninfinite-shear plateaus (η∞).',
 #        transform=plt.gca().transAxes, fontsize=12,
 #        bbox=dict(boxstyle='round,pad=0.5', facecolor='aliceblue', alpha=0.9),
 #        verticalalignment='top', horizontalalignment='right', fontweight='bold')

# Labels, Title, and Legend
plt.xlabel('Shear Rate, γ̇ (s⁻¹)', fontsize=13)
plt.ylabel('Viscosity, η (Pa·s)', fontsize=13)
plt.title('Carreau-Yasuda Model: Linear Scale Zoomed View', fontsize=16, pad=10)
plt.legend(loc='upper right', fontsize=11)
plt.grid(True, which="both", ls="-", alpha=0.5)

# Set the zoom window for the axes
plt.ylim(0.0, 0.105)
plt.xlim(min(strain_rates), max(strain_rates))


plt.tight_layout()
plt.show()

# --- Print Summary (same as before) ---
print("Model Parameters:")
print("-" * 25)
print(f"  η₀ (zero-shear viscosity): {eta_0} (fixed)")
print(f"  λ (time constant):         {lambda_} (fixed)")
print(f"  a (shape parameter):         {a} (fixed)")
print(f"  n (power-law index):       {n} (fixed)")
print(f"  η∞ (inf-shear viscosity):  {eta_inf_values} (varied)")
print("\nCarreau-Yasuda Formula: η(γ̇) = η∞ + (η₀-η∞)[1+(λγ̇)ᵃ]^((n-1)/a)")

In [ ]:

# Import necessary functions
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from jax_rheology.models import carreau_yasuda_viscosity

# Define the Power-Law model function for comparison
def power_law_viscosity(strain_rate, K, n):
    """Calculates viscosity using the Power-Law model: η = K * γ̇^(n-1)"""
    return K * strain_rate**(n - 1)

# --- Parameters ---
# Fixed Carreau-Yasuda parameters
eta_0 = 1.0     # Zero-shear viscosity (Pa·s)
lambda_ = 5.0   # Time constant (s)
a = 2.0         # Shape parameter
n = 0.5         # Fixed power-law index

# Variable parameter: eta_inf (infinite-shear viscosity)
eta_inf_values = np.array([0.01, 0.025, 0.05, 0.075])

# Create strain rate range (linear scale, focused on the transition)
strain_rates = jnp.linspace(20, 300, 1000)  # Shear rates from 20 to 300 s⁻¹

# --- Plotting ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(12, 8))
colors = plt.cm.plasma(np.linspace(0, 0.8, len(eta_inf_values)))

# 1. Plot Carreau-Yasuda curves for each eta_inf value
for i, eta_inf in enumerate(eta_inf_values):
    viscosity = carreau_yasuda_viscosity(strain_rates, eta_inf, eta_0, lambda_, n, a)
    plt.plot(strain_rates, viscosity, color=colors[i], linewidth=2.5,
             label=f'Carreau-Yasuda (η∞ = {eta_inf})')
    # Add a horizontal line for the plateau of each curve
    plt.axhline(eta_inf, color=colors[i], linestyle='--', alpha=0.6)

# 2. Plot the comparative Power-Law model (re-added)
# We set the consistency index K to match the Carreau-Yasuda model in the power-law region
K = eta_0 * (lambda_ ** (n - 1))
power_law_visc = power_law_viscosity(strain_rates, K, n)
plt.plot(strain_rates, power_law_visc, color='black', linestyle='-.', linewidth=2, 
           label=f'Power-Law (K={K:.2f}, n={n:.1f})')

# --- Formatting ---
# Labels, Title, and Legend
plt.xlabel('Shear Rate, γ̇ (s⁻¹)', fontsize=13)
plt.ylabel('Viscosity, η (Pa·s)', fontsize=13)
plt.title('Carreau-Yasuda Model vs. Power-Law: Linear Scale Zoomed View', fontsize=16, pad=10)
plt.legend(loc='upper right', fontsize=11)
plt.grid(True, which="both", ls="-", alpha=0.5)

# Set the zoom window for the axes
plt.ylim(0.0, 0.15)
plt.xlim(min(strain_rates), max(strain_rates))

plt.tight_layout()
plt.show()

# --- Print Summary ---
print("Model Parameters:")
print("-" * 25)
print(f"  η₀ (zero-shear viscosity): {eta_0} (fixed)")
print(f"  λ (time constant):         {lambda_} (fixed)")
print(f"  a (shape parameter):         {a} (fixed)")
print(f"  n (power-law index):       {n} (fixed)")
print(f"  η∞ (inf-shear viscosity):  {eta_inf_values} (varied)")
print("\nCarreau-Yasuda Formula: η(γ̇) = η∞ + (η₀-η∞)[1+(λγ̇)ᵃ]^((n-1)/a)")
print(f"Power-Law Formula:    η(γ̇) = K * γ̇^(n-1), with K = {K:.3f}")

In [ ]:
results = run_serpentine('newtonian', 1.0, pressure_gradient=1.0,)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def superellipse(a=2.0, b=1.0, m=6, npts=800):
    """
    Parametric superellipse (Lamé curve).
    
    Equation:
        |x/a|^m + |y/b|^m = 1

    Args:
        a (float): half-length along x (horizontal)
        b (float): half-thickness along y (vertical)
        m (float): shape exponent (m=2 ellipse, m>4 boxier with rounded corners)
        npts (int): number of points

    Returns:
        (x, y) arrays for one closed loop
    """
    theta = np.linspace(0, 2*np.pi, npts)
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    r = 1.0 / (np.abs(cos_t/a)**m + np.abs(sin_t/b)**m)**(1.0/m)
    x = r * cos_t
    y = r * sin_t
    return x, y

def plot_superellipse(a=2.0, b=1.0, m=6, ax=None, **kwargs):
    """
    Quick plotting wrapper.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(5,5))
    x, y = superellipse(a, b, m)
    ax.plot(x, y, **kwargs)
    ax.set_aspect("equal", adjustable="box")
    ax.axhline(0, color="k", lw=0.5)
    ax.axvline(0, color="k", lw=0.5)
    ax.set_title(f"Superellipse a={a}, b={b}, m={m}")
    return ax

# Example: looks like your sketch
if __name__ == "__main__":
    plot_superellipse(a=2.0, b=0.4, m=5, color="black")
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def hourglass(a=2.0, b=0.4, m=0.6, npts=400):
    """
    Symmetric 'hourglass' (concave) superellipse:
        |x/a|^m + |y/b|^m = 1  with 0 < m < 1.
    We build one quadrant and mirror to avoid any asymmetry.

    a: half-length in x
    b: half-thickness in y (controls throat)
    m: concavity exponent (0.3–0.9 gives nice pinches; smaller = pointier)
    """
    # x along the right half
    x = np.linspace(-a, a, npts)
    # clamp to avoid slight >1 due to float
    t = np.clip(1.0 - np.abs(x / a) ** m, 0.0, 1.0)
    y = b * t ** (1.0 / m)          # top branch
    return x,  y, -y                # x, y_top, y_bot

def plot_hourglass(a=2.0, b=0.4, m=0.6):
    x, yt, yb = hourglass(a, b, m)
    fig, ax = plt.subplots(figsize=(6,2))
    ax.plot(x, yt, 'k', lw=2)
    ax.plot(x, yb, 'k', lw=2)
    ax.axhline(0, color='0.6', lw=0.5)
    ax.axvline(0, color='0.6', lw=0.5)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(-a*1.05, a*1.05)
    ax.set_ylim(-b*1.2,  b*1.2)
    ax.set_title(f"Hourglass: a={a}, b={b}, m={m}")
    plt.show()

# Example (very close to your sketch):
if __name__ == "__main__":
    plot_hourglass(a=2.0, b=0.4, m=0.6)

In [ ]:
result = run_channel_constriction('carreau_yasuda', 0.02, 1.0, 5.0, 0.5, 2.0, pressure_gradient =5.0, preconditioner='none', dt = 1e-5, inner_steps = 400, outer_steps = 400)

In [ ]:
result = run_channel_constriction('carreau_yasuda', 0.02, 1.0, 5.0, 0.5, 2.0, pressure_gradient =1.0, preconditioner='none', dt = 1e-5, inner_steps = 400, outer_steps = 2000)

In [ ]:
# Use final_tbnn_params.pkl (not tree_def!)
results = run_tbnn_demo(params_path = 'tbnn_debug_results_constriction_new/iteration_12_20251008_050525/trajectory_data/final_tbnn_params.pkl',
                        dt = 2e-5,
                        inner_steps = 400,
                        outer_steps = 1500,
                        pressure_gradient = 10.0,
                        save_trajectory = False)

In [ ]:
run_channel_obstacle('carreau_yasuda', 0.02, 1.0, 5.0, 0.7, 2.0, pressure_gradient = 10.0, preconditioner='helmholtz', dt = 2e-5, outer_steps = 1500, inner_steps = 400)

In [ ]:
# Use final_tbnn_params.pkl (not tree_def!)
results = run_porous_media_tbnn_demo(params_path = 'tbnn_debug_results_constriction_new/iteration_12_20251008_050525/trajectory_data/final_tbnn_params.pkl',
                        dt = 1e-4,
                        inner_steps = 400,
                        outer_steps = 200,
                        pressure_gradient = 10.0,
                        save_trajectory = False)

In [ ]:
results = run_porous_media('carreau_yasuda', 0.02, 1.0, 5.0, 0.7, 2.0,
                           dt=1e-4,
                           inner_steps=400,
                           outer_steps=200,
                           pressure_gradient=10.0,
                           save_trajectory=False,
                           show_plots=True)

In [ ]:
# Compare TBNN against Carreau-Yasuda (CY as ground truth)
comparison = run_demo_comparison(
    ground_truth_model='carreau_yasuda',
    ground_truth_params=(0.02, 1.0, 5.0, 0.7, 2.0),
    comparison_model='tbnn',
    comparison_params=('tbnn_debug_results_constriction_new/iteration_12_20251008_050525/trajectory_data/final_tbnn_params.pkl',  42),
    domain_size=(256, 256),  # Controls simulation resolution
    dt=5e-5,
    inner_steps=400,
    outer_steps=500,
    pressure_gradient=7.5,
    num_bins=12,
    save_trajectory=False
)